In [ ]:
# !pip install marker

```
Notes for Mahdi: 

1. The following notebook is how I am extracting text from the .jpeg/.pdf photographs of the Wagner text
2. The version of python I am using = Python 3.10.13
3. Transcriptions are done in 3 parts:
    A. Extract text from a PDF containing a batch of Wagner passages using Marker
    B. Extract text individually from .jpeg version of Wagner passages using Base64 and LLMs
    C. Use LLM to synthesize both transcriptions

```

In [2]:
# General packages
import glob
import tqdm
import os
import pickle as pkl

## Marker (OCR library) Packages
from marker.config.parser import ConfigParser
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
import cgi

## OCR on base64 packages
import base64
from bs4 import BeautifulSoup
from typing import get_args, Literal, Union

# LLM/Agent packages
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from openai.types.chat.chat_completion_content_part_param import (
    ChatCompletionContentPartTextParam,
    ChatCompletionContentPartImageParam
)
from openai.types.chat.chat_completion_content_part_image_param import (
    ImageURL
)
import pydantic_ai.models as ai_models

# Directory to photo training set
photo_dir = "/Users/williamharrigan/Desktop/wagner_2"

## To import API keys
import sys
# sys.path.append('/Users/williamharrigan/Desktop/')

sys.path.append('/Users/williamharrigan/Desktop/UH/Year_3/semester_2/wagner/')

# Script containing API keys
import creds


In [4]:
## Path to .jpeg photos of Wagner passages
jpeg_files = glob.glob(f'{photo_dir}/*.jpeg')

## Path to .pdf version of batch of Wagner photos
wagner_pdf_batch = glob.glob(f'{photo_dir}/*.pdf')[0] # test batch contains 2 photos -> complete batch contains 50 photos
# wagner_pdf_batch = glob.glob(f'{photo_dir}/complete*.pdf')[0]
print('Marker photo: ', wagner_pdf_batch)

# Sort files to process alphabetically
# sorted_files = sorted(jpeg_files)
# print('Base64/OCR photo: ', sorted_files[0])

test_photo = '/Users/williamharrigan/Desktop/wagner_2/oleaceae.pdf'

Marker photo:  /Users/williamharrigan/Desktop/wagner_2/ochnaceae.pdf


## Marker OCR

In [7]:
## Set configuration for Marker text extraction

# Sets LLM component to use Gemini, but can use any available LLM
# config = {
#     "output_format": "json",
#     "use_llm": True,
#     "gemini_api_key": os.environ["GEMINI_API_KEY"], # API key
#     "gemini_model": "gemini-2.5-pro-preview-03-25", # LLM
#     "llm_service": "marker.services.gemini.GoogleGeminiService" # Set for converter
# }
config = {
    "output_format": "json",
    "use_llm": True,
    "openai_api_key": os.environ["OPENAI_API_KEY"],
    "openai_model": "o4-mini",
    "llm_service": "marker.services.openai.OpenAIService"
}

# Image parser
config_parser = ConfigParser(config)

# Default parameters for Marker OCR
converter = PdfConverter(
    config=config_parser.generate_config_dict(),
    artifact_dict=create_model_dict(),
    processor_list=config_parser.get_processors(),
    renderer=config_parser.get_renderer(),
    llm_service=config["llm_service"]  # Set in config (above)
)

Loaded layout model s3://layout/2025_02_18 on device mps with dtype torch.float16
Loaded texify model s3://texify/2025_02_18 on device mps with dtype torch.float16
Loaded recognition model s3://text_recognition/2025_02_18 on device mps with dtype torch.float16
Loaded table recognition model s3://table_recognition/2025_02_18 on device mps with dtype torch.float16
Loaded detection model s3://text_detection/2025_02_28 on device mps with dtype torch.float16
Loaded detection model s3://inline_math_detection/2025_02_24 on device mps with dtype torch.float16


In [6]:
## Run Marker OCR on batch -> output in .json format

rendered = converter(test_photo)

Recognizing layout: 100%|██████████| 1/1 [00:03<00:00,  3.40s/it]
LLM layout relabelling: 2it [00:03,  1.53s/it]
Recognizing Text: 100%|██████████| 8/8 [01:02<00:00,  7.85s/it]
Detecting bboxes: 0it [00:00, ?it/s]
LLMTableMergeProcessor running: 0it [00:00, ?it/s]
LLM processors running: 100%|██████████| 1/1 [00:03<00:00,  3.30s/it]


In [14]:
from bs4 import BeautifulSoup

marker_ocr_outputs = {}
total = []

def get_reading_order_text(blocks, column_split=0.5):
    """
    Sort blocks into reading order for a two-column layout.
    column_split: fractional x position dividing left/right columns (0.5 = midpoint).
    """
    if not blocks:
        return []

    # Find page width from the rightmost x1
    page_width = max(b["x1"] for b in blocks)
    split_x = page_width * column_split

    left_col  = [b for b in blocks if b["x0"] < split_x]
    right_col = [b for b in blocks if b["x0"] >= split_x]

    # Sort each column top-to-bottom by the top edge (y0)
    left_col.sort(key=lambda b: b["y0"])
    right_col.sort(key=lambda b: b["y0"])

    # Read left column first, then right
    return left_col + right_col

for i in range(len(rendered.children)):

    blocks = rendered.children[i].children

    block_list = []
    for block in blocks:
        text = BeautifulSoup(block.html, "html.parser").get_text().strip()
        if not text:
            continue

        # bbox = [x0, y0, x1, y1]
        block_list.append({
            "text": text,
            "x0": block.bbox[0],  # left edge
            "y0": block.bbox[1],  # top edge
            "x1": block.bbox[2],  # right edge
            "y1": block.bbox[3],  # bottom edge
        })

    ordered = get_reading_order_text(block_list, column_split=0.5)

    for block in ordered:
        total.append(block["text"])

full_text = "\n\n".join(total)

print("Marker Extracted Text:")
print(full_text)

Marker Extracted Text:
990

76. OLEACEAE Olive family

Trees or shrubs, sometimes climbing. Leaves simple to pinnately compound or trifoliolate, opposite or rarely alternate, covered with peltate glands especially on lower surface. sometimes with clusters of secretory hairs forming extrafloral nectaries, margins often entire, petiolate, stipules absent. Flowers perfect or occasionally unisexual (and then plants dioecious), actinomorphic, in basically cymose inflorescences, these often decussate, fasciculate, or paniculate, sometimes flowers solitary; calyx lobes 4(5), valvate, rarely calyx absent; corolla lobes 4(-12), valvate or imbricate, sometimes petals nearly distinct or absent; nectary disk absent, or sometimes present around base of ovary; stamens 2(4), inserted on corolla tube alternate with the lobes; anthers dithecal, oriented back-to-back, opening by longitudinal slits; ovary superior, 2-carpellate, with as many cells, placentation axile, ovules (1)2(4 to numerous) per cell,

In [17]:
plant_species = 'test'
marker_ocr_outputs[plant_species] = full_text

In [15]:
# ## Processing Marker Text Extractions

# # Store marker text extractions
# marker_ocr_outputs = {}

# # Iterate through rendered passages
# for i in range(len(rendered.children)):
    
#     plant_species = sorted_files[i].split('/')[-1].strip('.jpeg')
#     print(f"Plant: {plant_species}")
    
#     # Extract all text blocks from rendered
#     blocks = rendered.children[i].children

#     # Reorder extracted Wagner text using BeautifulSoup
#     block_list = []
#     for block in blocks:
#         text = BeautifulSoup(block.html, "html.parser").get_text()
#         block_list.append({
#             "text": text.strip(),
#             "x": block.bbox[2],  # left coordinate
#             "y": block.bbox[3]   # top coordinate
#         })
        
#     block_list.sort(key=lambda b: (b["x"], b["y"]))

#     print('Marker Extracted Text:')

#     # Concatenate Text
#     total = []

#     for block in block_list:
#         total.append(block["text"])
    
#     # Save to dictionary
#     marker_ocr_outputs[plant_species] = total
#     print(f"{total}\n")


## base64 OCR

In [23]:
# Async transcription function
async def transcribe_image_with_openai(image_path: str) -> str:
    with open(image_path, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode("utf-8")

    agent = Agent(
        model="openai:gpt-4o",
        result_type=str,
        system_prompt="You are a vision model capable of accurately performing OCR on an image",
    )

    image_param = ChatCompletionContentPartImageParam(
        type='image_url',
        image_url=ImageURL(url=f"data:image/png;base64,{base64_image}", detail='low')
    )

    message = [
        ChatCompletionContentPartTextParam(
            type="text",
            text="Convert the image to text. Don't miss any text and DO NOT ADD ANY TEXT that is not present in the image."
        ),
        image_param
    ]

    response = await agent.run(message)
    return response.data

In [34]:
async def transcribe_image_with_openai(image_path: str) -> str:
    with open(image_path, "rb") as f:
        base64_image = base64.b64encode(f.read()).decode("utf-8")

    # Fix 1: Use correct MIME type for .jpeg files
    mime_type = "image/jpeg"  # was "image/png" — must match actual file type

    image_param = ChatCompletionContentPartImageParam(
        type='image_url',
        image_url=ImageURL(
            url=f"data:{mime_type};base64,{base64_image}",
            detail='high'  # use 'high' for text-heavy scans
        )
    )

    message = [
        ChatCompletionContentPartTextParam(
            type="text",
            text="Transcribe all text in this image exactly as it appears."
        ),
        image_param
    ]

    response = await client.chat.completions.create(
        model="gpt-4o",  # Fix 2: Must be a vision-capable model (gpt-4o, gpt-4-turbo, gpt-4-vision-preview)
        messages=[{"role": "user", "content": message}],
        max_tokens=4096
    )

    return response.choices[0].message.content

await transcribe_image_with_openai('/Users/williamharrigan/Desktop/wagner_2/oleaceae.pdf')

NameError: name 'client' is not defined

In [31]:
## Run base64 OCR on .jpeg images of Wagner Passages

# Storage for Extracted Text
openai_transcriptions = {}

# Plant ID = format files are saved in = family_genus_species.jpeg
plant_id = 'test'

# Set image path
# image_path = f"{photo_dir}/{plant_id}.jpeg"

# Run transcription agent
openai_transcriptions[plant_id] = await transcribe_image_with_openai('/Users/williamharrigan/Desktop/wagner_2/oleaceae.pdf')

## Loop to process all Wagner .jpeg images

# for image_path in tqdm.tqdm(jpeg_files):
#     plant_id = image_path.split('/')[-1].strip('.jpeg') 
#     openai_transcriptions[plant_id] = await transcribe_image_with_openai(image_path)



NameError: name 'client' is not defined

In [21]:
## Create Agent to Synthesize base64 and Marker text extractions

create_final_transcription = Agent(
    model="openai:gpt-4o",
    result_type=str,
    system_prompt=(
        "You are a botanical plant expert. Given two OCR outputs of the same passage on Hawaiian plants, merge them into one accurate, complete transcription. Choose the most reliable text from each version."
    )
)

In [27]:
# Dictionary for final transcriptions
synthesized_transcriptions = {}

# Iterate through available transcriptions (jpeg_files, openai_transcriptions or marker_ocr_outputs)
for plant_id in tqdm.tqdm(openai_transcriptions.keys()): 

    # Get transcriptions from dictionary
    ocr_1 = openai_transcriptions[plant_id]
    ocr_2 = marker_ocr_outputs[plant_id]

    # Input to LLM
    og_transcriptions = f""" Here are the following two passages. It is very important to include page number and any other relevant information. Do not add any information that is not present in the transcriptions.
    OCR text 1:
    {ocr_1}

    OCR text 2:
    {ocr_2}
    """

    # Run LLM-agent to get synthesized Wagner passage text
    r = await create_final_transcription.run(og_transcriptions)
    
    # Save to dictionary
    synthesized_transcriptions[plant_id] = r.data
    
    
# Save file to pkl
# with open('wagner_transcriptions.pkl', 'wb') as f:
#     pkl.dump(synthesized_transcriptions, f)

100%|██████████| 1/1 [00:05<00:00,  5.01s/it]


In [24]:
for key, value in synthesized_transcriptions.items():
    print(key)
    print(value)

Acanthaceae_Dicliptera_chinensis
---

171

1. Dicliptera chinensis (L.) Juss. [Justicia chinensis L.] (nat)

Sprawling or decumbent perennial herbs; stems 2-7 dm long. Leaves green, lower surface slightly paler, ovate, 2.5-13.5 cm long, sparsely strigillose, especially on the veins, cystoliths prominent on upper surface; white raised streaks the size of the larger petals, 1-3.5 cm long. Flowers in axillary cymes, each one subtended by 2 green, ovate bracts of unequal size, the larger one ca. 12-14 mm long, the smaller one ca. 8-9 mm long, all bracts short-villous especially along the margins, the veins inconspicuous, pedicels 0.7-1 mm long; calyx lobes of unequal size, 5-17 mm long; corolla conspicuous, rose to purple, the throat with purple spot, 5-13 mm long. Capsules ovoid, 6-7 mm long, short-villous. Seeds 4, discoid. Native to tropical areas worldwide; in Hawai'i naturalized on O'ahu; in moist disturbed areas, at least on Kaua'i and O'ahu, but perhaps more widespread. Popenoe (192